# 14e The NSD one-session benchmark with nested, image-grouped calibration

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a, 10b and the four 12a notebooks.

Development analysis, not a prespecified test. 12b run again unchanged (it must reproduce 12b), plus every system with nested, image-grouped calibration
folds, and the person's own DINOv2 ridge model on the whole recording as a reference.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '14e-nestedbench', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"targets": ["subj01", "subj02", "subj05", "subj07"], "K": 200, "folds": 5, "seed": 13, "ridge_seed": 13, "n_subsets": 30, "subset_size": 300, "subset_seed": 401, "basis_images": 3000}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
common, cols = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
NEW = {}
for s in FIXED["targets"]:
    NEW[s] = dict(np.load(next(IN.rglob(f"12a_{s}.npz"))))
    if not (np.array_equal(NEW[s]["common_lh"], common["lh"]) and np.array_equal(NEW[s]["common_rh"], common["rh"])):
        raise ValueError(f"{s}: 12a used other vertices")
shared_ids = NEW[FIXED["targets"][0]]["shared_ids"]
ids, Y, pool_rows = {}, {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    pool_rows[s] = np.where(~np.isin(ids[s], shared_ids))[0]  # every shared image out of the donors' training
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[pool_rows[s]][:, cols[h][s]] for h in ("lh", "rh")], 1)
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float32), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]].astype(np.float64)
for s in FIXED["targets"]:
    missing = [int(i) for i in np.r_[NEW[s]["train_ids"], NEW[s]["test_ids"]] if int(i) not in FEATS["alexnet"][1]]
    if missing:
        raise ValueError(f"{s}: {len(missing)} images without features")
print(V, "vertices | donor pools", {s: len(pool_rows[s]) for s in SUBJECTS}, "| new people",
      {s: (len(NEW[s]["train_ids"]), len(NEW[s]["test_ids"])) for s in FIXED["targets"]}, "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def ols_coef(x, y):
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
def ridge_coef(x, y):
    m = RidgePerOutput(lm.ALPHAS, FIXED["folds"], FIXED["ridge_seed"]).fit(x, np.asarray(y, np.float32))
    print("   alpha median", float(np.median(m.best_alpha_)), "at grid edge", round(m.alpha_at_edge_, 3), flush=True)
    return np.vstack([(m.y_mean_ - m.x_mean_ @ m.coef_)[None], m.coef_]).astype(np.float32)
lin_pred = lambda beta, x: np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64)
DONORS = {"alexnet": ("alexnet", ols_coef), "dinov2": ("dinov2", ols_coef), "dinov2_ridge": ("dinov2", ridge_coef)}
BETA = {}
for kind, (f, fit) in DONORS.items():
    for s in SUBJECTS:
        BETA[(kind, s)] = fit(feats(f, ids[s][pool_rows[s]]), Y[s])
    print(kind, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
SUBSETS = [np.sort(np.random.default_rng([FIXED["subset_seed"], j]).permutation(n)[:FIXED["subset_size"]])
           for j, n in enumerate([len(NEW[FIXED["targets"][0]]["test_ids"])] * FIXED["n_subsets"])]

def zrows(a):
    a = np.asarray(a, np.float64)
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-30)

def retrieval(pred, meas, centred=False):
    # top-1 among 300, mean over the 30 subsets: image retrieval (each measured pattern picks among 300 predictions)
    # and brain retrieval (each prediction picks among 300 measured patterns); Pearson over vertices
    if centred:  # secondary: centred over the test images, which uses unlabelled test data
        pred, meas = pred - pred.mean(0), meas - meas.mean(0)
    p, m = zrows(pred), zrows(meas)
    img, brain = [], []
    for S in SUBSETS:
        r = m[S] @ p[S].T / p.shape[1]
        img.append((r.argmax(1) == np.arange(len(S))).mean() * 100)
        brain.append((r.argmax(0) == np.arange(len(S))).mean() * 100)
    return float(np.mean(img)), float(np.mean(brain))

def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        pred = fit_predict(np.setdiff1d(np.arange(k), held), held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

pool_all = np.unique(np.concatenate([ids[s][pool_rows[s]] for s in SUBJECTS]))
basis_ids = np.random.default_rng(FIXED["seed"]).choice(pool_all, FIXED["basis_images"], replace=False)  # images only
rows = []
for s in FIXED["targets"]:
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    tr_ids, te_ids = NEW[s]["train_ids"], NEW[s]["test_ids"]
    y_k = NEW[s]["train_y"].astype(np.float64)
    meas = NEW[s]["test_mean"].astype(np.float64)
    ceil = NEW[s]["ceiling"].astype(np.float64)
    k, d = len(tr_ids), FIXED["seed"]
    mc = meas - meas.mean(0)
    def record(system, kind, pred, labelled):
        pc = pred - pred.mean(0)
        rv = (pc * mc).sum(0) / np.sqrt((pc ** 2).sum(0) * (mc ** 2).sum(0) + 1e-30)
        img, brain = retrieval(pred, meas)
        img_c, brain_c = retrieval(pred, meas, centred=True)
        rows.append({"subject": s, "system": system, "donors": kind, "labelled": labelled, "enc_r": float(rv.mean()),
                     "enc_r_over_ceiling": float(np.median(rv / np.maximum(ceil, 1e-3))), "image_ret": img, "brain_ret": brain,
                     "image_ret_centred": img_c, "brain_ret_centred": brain_c})
        print(f"  {s} {system:10s} {kind:13s} r {rv.mean():.3f} | image {img:.1f} brain {brain:.1f} (centred {img_c:.1f} / {brain_c:.1f})", flush=True)

    groups = np.asarray(tr_ids)
    labels, inverse = np.unique(groups, return_inverse=True)
    row_folds = np.array_split(np.random.default_rng(d).permutation(k), FIXED["folds"])  # 12b's folds, over trials
    img_folds = [np.flatnonzero(np.isin(inverse, g))
                 for g in np.array_split(np.random.default_rng(d).permutation(len(labels)), FIXED["folds"])]
    def oof_of(fit_predict, folds):
        out = np.empty((k, V), np.float32)
        for held in folds:
            out[held] = fit_predict(np.setdiff1d(np.arange(k), held), held)
        return out
    parts = {}  # only the parts the combination uses: the map on DINOv2 borrowing (both donors), own AlexNet model
    for kind, (f, _) in DONORS.items():
        x_k, x_t, x_b = feats(f, tr_ids), feats(f, te_ids), feats(f, basis_ids)
        def avg_of(x):
            acc = np.zeros((len(x), V))
            for dn in donors:
                acc += lin_pred(BETA[(kind, dn)], x)
            return acc / len(donors)
        a_k, a_t = avg_of(x_k), avg_of(x_t)
        mean, B = lm.prediction_basis(avg_of(x_b), FIXED["K"])
        record("borrowing", kind, a_t - a_k.mean(0), 0)
        for how, grp, folds in (("12b", None, row_folds), ("nested", groups, img_folds)):
            fit = lambda tr: lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d, groups=None if grp is None else grp[tr])
            m_t = lm.apply_lowrank_map(fit(np.arange(k)), a_t, mean, B)
            record("map" if how == "12b" else "map_grouped", kind, m_t, k)
            if kind != "alexnet":
                parts[(how, "map", kind)] = (oof_of(lambda tr, he: lm.apply_lowrank_map(fit(tr), a_k[he], mean, B), folds), m_t.astype(np.float32))
        if kind in ("alexnet", "dinov2"):
            own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_k, y_k)
            record("own_ridge", f, own.predict(x_t), k)
            own_g = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_k, y_k, groups)
            record("own_ridge_grouped", f, own_g.predict(x_t), k)
            if f == "alexnet":
                # 12b: penalties chosen once on all k trials, reused in every fold; nested: chosen again inside each fold
                parts[("12b", "own", f)] = (oof_of(lambda tr, he: own.refit(x_k[tr], y_k[tr]).predict(x_k[he]), row_folds), own.predict(x_t).astype(np.float32))
                parts[("nested", "own", f)] = (oof_of(lambda tr, he: RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_k[tr], y_k[tr], groups[tr]).predict(x_k[he]), img_folds),
                                               own_g.predict(x_t).astype(np.float32))
            record("own_full_ols", f, lin_pred(BETA[(kind, s)], x_t), -1)  # 12b's reference: least squares, all their images
        if kind == "dinov2_ridge":
            record("own_full_ridge", "dinov2", lin_pred(BETA[("dinov2_ridge", s)], x_t), -1)  # ridge, all their images
        del a_k, a_t
    for kind in ("dinov2", "dinov2_ridge"):
        for how, name in (("12b", "mixed"), ("nested", "mixed_nested")):
            chosen = [(how, "map", kind), (how, "own", "alexnet")]
            w = nc.stack_fit([parts[c][0].astype(np.float64) for c in chosen], y_k)
            record(name, kind, nc.stack_apply(w, [parts[c][0].astype(np.float64) for c in chosen],
                                              [parts[c][1].astype(np.float64) for c in chosen], y_k.mean(0)), k)
    del parts
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "14e_scores.csv", index=False)
    print(s, "k", k, "unique images", len(labels), f"| {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
frame = pd.DataFrame(rows)
METRICS = ("enc_r", "image_ret", "brain_ret", "image_ret_centred", "brain_ret_centred")
res = {"post_hoc": "development: 12b with nested, image-grouped calibration and a ridge whole-recording reference",
       "systems": {}, "train_trials": {s: int(len(NEW[s]["train_ids"])) for s in FIXED["targets"]},
       "train_images": {s: int(len(np.unique(NEW[s]["train_ids"]))) for s in FIXED["targets"]}}
def per(system, kind, m="image_ret"):
    return frame[(frame.system == system) & (frame.donors == kind)].set_index("subject")[m].reindex(FIXED["targets"])
for (system, kind), g in frame.groupby(["system", "donors"]):
    res["systems"][f"{system}|{kind}"] = {m: float(per(system, kind, m).mean()) for m in METRICS}
    res["systems"][f"{system}|{kind}"]["per_person_image_ret"] = per(system, kind).round(2).to_dict()
def diff(a, b, m="image_ret"):
    x = per(*a, m) - per(*b, m)
    return {"mean": float(x.mean()), "positive": int((x > 0).sum()), "n": int(x.notna().sum()), "per_person": x.round(2).to_dict()}
res["contrasts"] = {}
for m in ("image_ret", "brain_ret"):
    for kind in ("dinov2", "dinov2_ridge"):
        res["contrasts"][f"mixed_nested - mixed|{kind}|{m}"] = diff(("mixed_nested", kind), ("mixed", kind), m)
        res["contrasts"][f"map_grouped - map|{kind}|{m}"] = diff(("map_grouped", kind), ("map", kind), m)
    for f in ("alexnet", "dinov2"):
        res["contrasts"][f"own_ridge_grouped - own_ridge|{f}|{m}"] = diff(("own_ridge_grouped", f), ("own_ridge", f), m)
    res["contrasts"][f"own_full_ridge - own_full_ols|dinov2|{m}"] = diff(("own_full_ridge", "dinov2"), ("own_full_ols", "dinov2"), m)
# share of the distance from borrowing to the whole-recording model that the one-session combination closes
res["distance_closed"] = {}
for ref in ("own_full_ols", "own_full_ridge"):
    for name in ("mixed", "mixed_nested"):
        lo, hi, v = per("borrowing", "dinov2").mean(), per(ref, "dinov2").mean(), per(name, "dinov2").mean()
        res["distance_closed"][f"{name}|{ref}"] = float((v - lo) / (hi - lo))
res["check_12b"] = {"map|dinov2": res["systems"]["map|dinov2"]["image_ret"], "expected_map": 61.4944,
                    "mixed|dinov2": res["systems"]["mixed|dinov2"]["image_ret"], "expected_mixed": 64.52,
                    "own_full_ols|dinov2": res["systems"]["own_full_ols|dinov2"]["image_ret"], "expected_own_full": 70.7778}
(C.METRICS_DIR / "14e_summary.json").write_text(json.dumps(res, indent=2))
print("14E_SUMMARY_JSON " + json.dumps(res))